# Multi-Task Repository Evolution with RHO and HELIX

A one-shot coding agent is enough for a prescriptive one-line repair. This experiment asks a harder question: can reflective evolution improve a repository containing **three interacting robot policies** while preserving a working regression guard?

The seed repository contains authentic Gemma E4B CaP-X policies for cube stacking, spill wiping, and cube lifting. Qwen3-Coder proposes competing repository mutations over **two generations**. HELIX applies a strict training gate, evaluates survivors on all validation tasks, and retains per-task winners on an instance Pareto frontier.

The goal is not necessarily one universal winner. A stack specialist and a wipe specialist are useful evidence when each protects cube lift; generation 2 can select or merge those frontier members.

> This is a bounded teaching experiment, not a reproduction of the HELIX or GEPA papers' full runs.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(r"""
## What makes this an evolution problem?

- **Multiple files:** task policies live in separate modules and may share geometry/runtime helpers.
- **Conflicting evidence:** each mutation sees one training task, while promotion requires full validation.
- **Candidate competition:** repositories can win different validation examples instead of collapsing to one scalar rank.
- **Regression protection:** cube lift is already strong and remains a validation-only guard.
- **Reflective generations:** generation 2 receives evaluator traces and selects from the frontier built by generation 1.

A coding agent still performs each mutation. HELIX contributes the experimental loop around that agent: sampling, evaluation, strict minibatch acceptance, Pareto retention, lineage, and optional crossover.
"""))

In [ ]:
import hashlib
import json
import sys
from pathlib import Path
from time import perf_counter

from IPython.display import Markdown, display

sys.path.insert(0, "/ryzers")
import rho_demo
import rho_multitask_demo as experiment
from capx_demo import show_rollout_grid

EXPERIMENT_ROOT = Path("/tmp/rho_multitask_notebook")
rho_demo.VIDEO_ROOT = EXPERIMENT_ROOT / "videos"
GENERATIONS = 2
RHO_TIMEOUT_SECONDS = 1200
ROLLOUT_TIMEOUT_SECONDS = 180
HIDDEN_TRIALS = {
    "cube_stack": [8740, 9351, 6027, 7419, 4883],
    "spill_wipe": [1854, 3167, 5279, 6481, 7903],
    "cube_lift": [2195, 3417, 5639, 6851, 9073],
}
HIDDEN_ROLLOUT_COUNT = sum(len(trials) for trials in HIDDEN_TRIALS.values())

print("Lemonade loader alias:", experiment.DEFAULT_MODEL)
print("OpenCode API model:", experiment.opencode_model_id())
print("Generations:", GENERATIONS, "· proposal slots per generation: 2")
print("Frozen hidden rollout trials:", HIDDEN_TRIALS)
print("Final hidden rollout videos:", HIDDEN_ROLLOUT_COUNT, "(5 per task)")

In [ ]:
display(Markdown(r"""
## The HELIX mapping to GEPA

1. **Evaluate evidence:** run a parent repository on one sampled training task.
2. **Reflect and mutate:** Qwen3-Coder receives API contracts plus evaluator diagnostics, then edits `solver/`.
3. **Strict train gate:** the child must beat its parent on that sampled task.
4. **Full validation:** only gate survivors run stack, wipe, and lift validation.
5. **Instance frontier:** HELIX retains the best repository for each validation example.
6. **Select or merge:** the next generation chooses frontier parents; crossover is eligible when specialists overlap on the lift guard.

Strict acceptance and Pareto retention answer different questions. The train gate asks whether a child improved the evidence it saw. The frontier asks which validated candidates remain useful for at least one task.
"""))

In [ ]:
import os

MOCK_MODE = os.environ.get("RHO_MULTITASK_MOCK") == "1"
experiment._safe_reset(EXPERIMENT_ROOT)
setup_started = perf_counter()
if MOCK_MODE:
    servers = []
    rho_demo.LAST_SERVICE_TIMING = {"lemonade_seconds": 0.0, "robotics_seconds": 0.0}
    print(experiment.MOCK_LABEL, "— no model or simulator service was used")
else:
    servers = experiment.ensure_services()
SETUP_SECONDS = perf_counter() - setup_started

ROOT = experiment.prepare_workshop(
    EXPERIMENT_ROOT / "candidate",
    model=experiment.DEFAULT_MODEL,
    generations=GENERATIONS,
)
MANIFEST = json.loads((ROOT / "scenarios.json").read_text())
PROVENANCE = json.loads((ROOT / "provenance.json").read_text())

print(f"Setup wall time: {SETUP_SECONDS:.1f}s")
print("Candidate repository:", ROOT)
print("Train examples:", MANIFEST["splits"]["train"])
print("Validation examples:", MANIFEST["splits"]["val"])
print("Hidden rollouts exposed to evolution:", MANIFEST["hidden_rollouts_exposed_to_evolution"])

In [ ]:
for task in ("cube_stack", "spill_wipe", "cube_lift"):
    policy = ROOT / "solver" / "tasks" / f"{task}.py"
    digest = hashlib.sha256(policy.read_bytes()).hexdigest()[:12]
    source = PROVENANCE["policies"][task]
    print(f"\n--- {task} · seed sha256 {digest} ---")
    print(f"Gemma source: {source['source']} · source trial {source['source_trial']}")
    print(policy.read_text())

print("Editable shared modules:", ["solver/geometry.py", "solver/runtime.py"])
print("Protected experiment files:", [
    "probe.py", "helix.toml", "opencode.json", "CONTRACT.md",
    "scenarios.json", "provenance.json",
])

In [ ]:
display(Markdown(r"""
## Establish the three-task baseline

The seen validation vector has three independent keys: `stack_val`, `wipe_val`, and `lift_guard_val`. HELIX uses those examples to maintain the instance frontier; it does not average them into a single winner.

The evaluator reports both `raw_reward` and deployable `reward`. Partial motion may produce raw reward, but an exception, timeout, or sandbox failure forces deployable reward to zero.

We also freeze one later trial per task now. Those hidden results stay outside the candidate repository and are never passed to HELIX. They are reused only after evolution for a paired deployment check.
"""))

In [ ]:
import pandas as pd

RESULT_KEYS = (
    "scenario_id", "task", "trial", "reward", "raw_reward",
    "task_completed", "timed_out", "stderr", "traceback",
    "feedback", "video", "elapsed_seconds",
)


def compact(result):
    return {key: result.get(key) for key in RESULT_KEYS}


def evaluate_split(policy_root, split):
    names = json.loads((policy_root / "scenarios.json").read_text())["splits"][split]
    results = []
    for index, (scenario_id, scenario) in enumerate(
        experiment.resolve_scenarios(policy_root, split, [str(i) for i in range(len(names))]),
        start=1,
    ):
        result = compact(experiment.score_scenario(
            policy_root,
            split,
            scenario_id,
            scenario,
            timeout_seconds=ROLLOUT_TIMEOUT_SECONDS,
        ))
        results.append(result)
        print(
            f"{index}/{len(names)} {scenario_id}: reward={float(result['reward'] or 0):.3f} · "
            f"raw={float(result['raw_reward'] or 0):.3f} · completed={result['task_completed']} · "
            f"{float(result['elapsed_seconds'] or 0):.1f}s"
        )
    return results


def task_frame(results):
    return pd.DataFrame([
        {
            "scenario": item.get("scenario_id", "unknown"),
            "task": item.get("task", "unknown"),
            "reward": float(item.get("reward") or 0),
            "raw reward": float(item.get("raw_reward") or 0),
            "completed": bool(item.get("task_completed")),
            "execution failed": bool(
                item.get("stderr") or item.get("traceback") or item.get("timed_out")
            ),
            "seconds": round(float(item.get("elapsed_seconds") or 0), 1),
        }
        for item in results
    ]).set_index("scenario")

In [ ]:
baseline_started = perf_counter()
BASELINE_VALIDATION = evaluate_split(ROOT, "val")
HIDDEN_BEFORE = experiment.hidden_rollouts(
    ROOT,
    trials=HIDDEN_TRIALS,
    capture=False,
)
BASELINE_SECONDS = perf_counter() - baseline_started

display(Markdown("### Seen validation baseline"))
display(task_frame(BASELINE_VALIDATION))
display(Markdown("### Frozen hidden baseline (not exposed to evolution)"))
display(task_frame(HIDDEN_BEFORE))

### Different tasks fail for different reasons

The stack policy misuses a flat pose returned by the simulator. The wipe policy can earn substantial raw reward yet continue sending blocking actions after the episode terminates. Cube lift provides a working guard against broad edits that damage grasp-and-lift behavior.

The mutation prompt does not prescribe either repair. Each proposal receives its sampled task's evaluator trace, API contracts, and repository context. This distinction matters: generation 1 can produce two valid specialists from different evidence.

In [ ]:
for item in BASELINE_VALIDATION:
    print(f"\n===== {item['scenario_id']} / {item['task']} =====")
    print(item["feedback"][-1800:])

repo_text = "\n".join(
    path.read_text(errors="ignore")
    for path in ROOT.rglob("*")
    if path.is_file() and ".git" not in path.parts
)
for task_trials in HIDDEN_TRIALS.values():
    for hidden_trial in task_trials:
        assert str(hidden_trial) not in repo_text
print("\nSplit isolation check passed: all hidden trial IDs are absent from the candidate repository.")

## Run two generations

The fixed configuration uses two proposal slots per generation, one mutation per selected parent, minibatch size 1, and serialized simulator workers. With RNG seed 29, the first two slots cover wipe and stack rather than duplicating one failure mode. The perfect-score threshold is deliberately 1.1 while rewards are bounded by 1.0, preventing a lucky generation-1 candidate from skipping the required second generation.

A proposal that does not strictly improve its sampled training task stops at the gate. A survivor incurs the more expensive full three-task validation. `frontier_type = "instance"` then records per-example winners. Merge is enabled with overlap floor 1, so generation-1 specialists that both preserve lift can become crossover parents.

The notebook displays every official HELIX evaluation separately from any self-check requested by the coding agent.

In [ ]:
import random

slot_order = list(MANIFEST["splits"]["train"])
random.Random(29).shuffle(slot_order)
print("Deterministic generation-1 sampled tasks:", slot_order)
print("HELIX config:")
for line in (ROOT / "helix.toml").read_text().splitlines():
    if any(key in line for key in (
        "max_generations", "perfect_score_threshold",
        "num_parallel_proposals", "minibatch_size", "acceptance_criterion",
        "frontier_type", "merge_enabled", "merge_val_overlap_floor",
        "max_evaluations",
    )):
        print(" ", line)

evolution_started = perf_counter()
if MOCK_MODE:
    LIVE_RUN = experiment.materialize_mock_evolution(ROOT)
    print(experiment.MOCK_LABEL, "— synthetic lineage is for notebook validation only")
else:
    LIVE_RUN = experiment.run_helix(
        ROOT,
        generations=GENERATIONS,
        timeout_seconds=RHO_TIMEOUT_SECONDS,
    )
EVOLUTION_SECONDS = perf_counter() - evolution_started
if LIVE_RUN.returncode != 0:
    raise RuntimeError(LIVE_RUN.stdout[-4000:])
print(f"HELIX completed in {EVOLUTION_SECONDS:.1f}s")

## Read the evolution, not just the final score

The next view joins HELIX state with candidate prompts, lineage, worktree diffs, train-gate outcomes, validation vectors, frontier keys won, parent IDs, and merge ancestry.

A mutation without a full-validation artifact was rejected or failed before promotion. A mutated candidate with a validation vector passed the strict minibatch gate; merge candidates use their separate merge-validation gate. Either may still lose every frontier key. Generation-1 candidates that win stack and wipe separately are specialists, not failed universal solutions.

For deployment we select the frontier member with the strongest difficult-task coverage among candidates that do not regress the lift guard.

In [ ]:
FRONTIER = experiment.frontier_summary(ROOT)
LESSON = experiment.evolution_lesson(FRONTIER)
LINEAGE = FRONTIER["lineage"]

frontier_rows = []
for candidate_id, candidate in FRONTIER["candidates"].items():
    frontier_rows.append({
        "candidate": candidate_id,
        **candidate["scores"],
        "keys won": ", ".join(candidate["wins"]),
        "retained": candidate["frontier"],
    })
display(pd.DataFrame(frontier_rows).set_index("candidate").fillna(0))

display(pd.DataFrame([
    {
        "candidate": item["id"],
        "generation": item["generation"],
        "operation": item["operation"],
        "parents": ", ".join(item["parents"]),
        "sampled train task": item["sampled_train_task"],
        "changed files": ", ".join(item["changed_files"]),
        "gate": item["gate_result"],
        "validation vector": item["validation_vector"],
    }
    for item in LINEAGE
]).set_index("candidate"))

print("Specialist/frontier lesson:", json.dumps(LESSON, indent=2))
print("Merge ancestry:", FRONTIER["merge_ancestry"])
print("Merge-output dedup ledger:", FRONTIER["merge_output_dedup_triplets"])

seed_lift = FRONTIER["candidates"].get("g0-s0", {}).get("scores", {}).get("lift_guard_val", 0.0)
eligible = {
    candidate_id: candidate
    for candidate_id, candidate in FRONTIER["candidates"].items()
    if candidate["scores"].get("lift_guard_val", 0.0) >= seed_lift
}
BEST_ID = max(
    eligible,
    key=lambda candidate_id: (
        eligible[candidate_id]["scores"].get("stack_val", 0.0)
        + eligible[candidate_id]["scores"].get("wipe_val", 0.0),
        sum(eligible[candidate_id]["scores"].values()),
        candidate_id,
    ),
)
EVOLVED_ROOT = ROOT / ".helix" / "worktrees" / BEST_ID
print("Selected frozen deployment repository:", BEST_ID, EVOLVED_ROOT)

for item in LINEAGE:
    if item["generation"] == 1 and item["changed_files"]:
        parent_root = ROOT / ".helix" / "worktrees" / item["parent"]
        child_root = ROOT / ".helix" / "worktrees" / item["id"]
        print(f"\n===== {item['id']} specialist diff =====")
        print(rho_demo.source_diff(parent_root, child_root))

In [ ]:
hidden_started = perf_counter()
HIDDEN_AFTER = experiment.hidden_rollouts(
    EVOLVED_ROOT,
    trials=HIDDEN_TRIALS,
    capture=True,
)
HIDDEN_AFTER_SECONDS = perf_counter() - hidden_started
for item in HIDDEN_AFTER:
    if item.get("video"):
        video_path = Path(item["video"]).resolve()
        item["video_relative"] = (
            str(video_path.relative_to(EXPERIMENT_ROOT.resolve()))
            if video_path.is_relative_to(EXPERIMENT_ROOT.resolve())
            else str(video_path)
        )

before_hidden = task_frame(HIDDEN_BEFORE).add_prefix("before ")
after_hidden = task_frame(HIDDEN_AFTER).add_prefix("after ")
display(before_hidden.join(after_hidden))
for task in ("cube_stack", "spill_wipe", "cube_lift"):
    display(Markdown(f"### {task.replace('_', ' ').title()} — five frozen rollouts"))
    show_rollout_grid([item for item in HIDDEN_AFTER if item["task"] == task])

HIDDEN_BEFORE_SUMMARY = experiment.summarize_rollouts(HIDDEN_BEFORE)
HIDDEN_AFTER_SUMMARY = experiment.summarize_rollouts(HIDDEN_AFTER)
summary_rows = []
for task in ("cube_stack", "spill_wipe", "cube_lift"):
    before = HIDDEN_BEFORE_SUMMARY[task]
    after = HIDDEN_AFTER_SUMMARY[task]
    summary_rows.append({
        "task": task,
        "rollouts": after["rollouts"],
        "completed before": before["completed"],
        "completed after": after["completed"],
        "completion rate before": before["completion_rate"],
        "completion rate after": after["completion_rate"],
        "mean reward before": before["mean_reward"],
        "mean reward after": after["mean_reward"],
        "execution failures after": after["execution_failures"],
    })
display(pd.DataFrame(summary_rows).set_index("task"))

lift_policy_unchanged = (
    hashlib.sha256((ROOT / "solver" / "tasks" / "cube_lift.py").read_bytes()).hexdigest()
    == hashlib.sha256((EVOLVED_ROOT / "solver" / "tasks" / "cube_lift.py").read_bytes()).hexdigest()
)
SUCCESS_CRITERION = experiment.hidden_success_criterion(
    HIDDEN_BEFORE_SUMMARY,
    HIDDEN_AFTER_SUMMARY,
    lift_policy_unchanged=lift_policy_unchanged,
)
hard_before = SUCCESS_CRITERION["hard_task_mean_reward_before"]
hard_after = SUCCESS_CRITERION["hard_task_mean_reward_after"]
lift_preserved = SUCCESS_CRITERION["lift_guard_preserved"]
print(
    "Hidden difficult-task completions:",
    SUCCESS_CRITERION["hard_task_completed_before"],
    "→",
    SUCCESS_CRITERION["hard_task_completed_after"],
    f"of {sum(HIDDEN_BEFORE_SUMMARY[task]['rollouts'] for task in ('cube_stack', 'spill_wipe'))}",
)
print("Hidden difficult-task mean reward:", f"{hard_before:.3f}", "→", f"{hard_after:.3f}")
print(
    "Hidden lift completion rate:",
    f"{SUCCESS_CRITERION['lift_completion_rate_before']:.0%}",
    "→",
    f"{SUCCESS_CRITERION['lift_completion_rate_after']:.0%}",
)
print("Lift policy unchanged:", lift_policy_unchanged)
print("Hidden lift guard preserved:", lift_preserved)
print("Workshop success criterion met:", SUCCESS_CRITERION["met"])

In [ ]:
service_timing = rho_demo.LAST_SERVICE_TIMING
agent_total = sum(item["agent_metrics"]["total_seconds"] for item in LINEAGE)
prompt_wait = sum(item["agent_metrics"]["prompt_wait_seconds"] for item in LINEAGE)
text_generation = sum(item["agent_metrics"]["text_generation_seconds"] for item in LINEAGE)
validation_simulator = sum(item["validation_simulator_seconds"] for item in LINEAGE)

print("Timing breakdown")
print(f"  Qwen model load:              {service_timing.get('lemonade_seconds', 0):7.1f}s")
print(f"  perception/control setup:     {service_timing.get('robotics_seconds', 0):7.1f}s")
print(f"  baseline + hidden rollouts:    {BASELINE_SECONDS:7.1f}s")
print(f"  HELIX total evolution:         {EVOLUTION_SECONDS:7.1f}s")
print(f"    OpenCode agent event span:   {agent_total:7.1f}s")
print(f"    estimated prompt wait:       {prompt_wait:7.1f}s")
print(f"    emitted text generation:     {text_generation:7.1f}s")
print(f"    retained validation sims:    {validation_simulator:7.1f}s")
print(f"  hidden video rollouts:         {HIDDEN_AFTER_SECONDS:7.1f}s")
print("\nPrompt/generation fields are OpenCode event estimates; simulator and total wall times are measured directly.")

In [ ]:
REPORT_PATH = EXPERIMENT_ROOT / "multitask_helix_report.json"
policy_hashes = {
    task: {
        "seed": hashlib.sha256((ROOT / "solver" / "tasks" / f"{task}.py").read_bytes()).hexdigest(),
        "selected": hashlib.sha256((EVOLVED_ROOT / "solver" / "tasks" / f"{task}.py").read_bytes()).hexdigest(),
    }
    for task in ("cube_stack", "spill_wipe", "cube_lift")
}
REPORT = {
    "schema_version": "rho-multitask-helix-report/v2",
    "mode": "mock_static_contract" if MOCK_MODE else "live_capx",
    "recorded_fallback": bool(MOCK_MODE),
    "mutation_model_loader_alias": experiment.DEFAULT_MODEL,
    "mutation_model_api_id": experiment.opencode_model_id(),
    "seed_model": PROVENANCE["seed_model"],
    "generations": GENERATIONS,
    "proposal_slots_per_generation": 2,
    "manifest": MANIFEST,
    "provenance": PROVENANCE,
    "policy_sha256": policy_hashes,
    "baseline_validation": BASELINE_VALIDATION,
    "frontier": FRONTIER,
    "lesson": LESSON,
    "selected_candidate": BEST_ID,
    "hidden_trials_used_by_evolution": False,
    "hidden_rollouts_per_task": 5,
    "hidden_before": HIDDEN_BEFORE,
    "hidden_after": HIDDEN_AFTER,
    "hidden_before_summary": HIDDEN_BEFORE_SUMMARY,
    "hidden_after_summary": HIDDEN_AFTER_SUMMARY,
    "success_criterion": SUCCESS_CRITERION,
    "timing": {
        "setup_seconds": SETUP_SECONDS,
        "service_setup": service_timing,
        "baseline_seconds": BASELINE_SECONDS,
        "evolution_seconds": EVOLUTION_SECONDS,
        "hidden_after_seconds": HIDDEN_AFTER_SECONDS,
        "agent_event_span_seconds": agent_total,
        "estimated_prompt_wait_seconds": prompt_wait,
        "emitted_text_generation_seconds": text_generation,
        "retained_validation_simulator_seconds": validation_simulator,
    },
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(REPORT, indent=2) + "\n")
print("Saved provenance-rich report:", REPORT_PATH)

display(Markdown(f"""
## Interpretation

- **Multi-key frontier formed:** `{LESSON['multi_key_frontier']}`
- **Stack specialists:** `{LESSON['stack_specialists']}`
- **Wipe specialists:** `{LESSON['wipe_specialists']}`
- **Broad candidates:** `{LESSON['broad_candidates']}`
- **Merge attempted:** `{LESSON['merge_attempted']}`
- **Selected deployment:** `{BEST_ID}`
- **Hidden evaluation:** `5` frozen trials per task (`15` final videos)
- **Hidden hard-task completions:** `{SUCCESS_CRITERION['hard_task_completed_before']}` → `{SUCCESS_CRITERION['hard_task_completed_after']}`
- **Hidden hard-task mean reward:** `{hard_before:.3f}` → `{hard_after:.3f}`
- **Hidden lift guard preserved:** `{lift_preserved}`
- **Workshop success criterion met:** `{SUCCESS_CRITERION['met']}`

A universal three-task winner is desirable but not required for the Pareto lesson. The success criterion now requires a completion gain or a meaningful aggregate reward gain across ten difficult-task rollouts, plus an unchanged lift policy whose five-rollout completion and reward stay within a one-rollout noise tolerance. Tiny non-completion reward changes do not count as success.
"""))

## What this notebook demonstrates

RHO evolves a versioned software artifact, not model weights. HELIX supplies the GEPA-style outer loop around a coding agent:

**diagnostics → reflective mutation → strict train gate → full validation → per-instance Pareto retention → frontier parent selection or merge → frozen deployment**

The important result is the evidence trail: sampled task, candidate ancestry, changed files, gate outcome, validation vector, frontier keys, merge ancestry, hidden rollouts, and measured timing. This is materially different from asking an agent for one known edit.

The disposable repository, HELIX worktrees, evaluator artifacts, and report remain under `/tmp/rho_multitask_notebook/`. Call `rho_demo.stop_owned_services()` when finished.

## What to expect—and what not to claim

Generation 1 should sample both hard-task paths. Depending on physical nondeterminism and mutation quality, one or both specialists may pass the strict gate. Generation 2 should show frontier-based parent selection or a merge attempt when specialist overlap permits it.

Do not claim success merely because code changed or raw reward rose. Report deployable reward, completion, execution failures, lift regression, and hidden-trial videos. If no mutation beats the gate, keep that rejection visible instead of substituting a recorded success.

For a deeper frontier example with grouped tasks and more candidates, continue to [`temp_evolving_rai.ipynb`](./temp_evolving_rai.ipynb). The LocalInference `README.md` distinguishes this live CaP-X repository evolution from the larger RAI experiment.